In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_curve, average_precision_score, classification_report)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import joblib
import warnings
import os
from collections import Counter
from scipy.stats import randint, uniform, wilcoxon
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:

File1 = pd.read_csv('C:/Users/User/Downloads/river-historical-1986_2020-ens.csv', encoding='latin1')
File1.head()

C:\Users\User\AppData\Local\Temp\ipykernel_15684\1757649013.py:1: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  File1 = pd.read_csv('C:/Users/User/Downloads/river-historical-1986_2020-ens.csv', encoding='latin1')


,Water Control Zone,River,Station,Dates,Sample No,5-Day Biochemical Oxygen Demand (mg/L),Aluminium (µg/L),Ammonia-Nitrogen (mg/L),Anionic Surfactants (as Manoxol OT) (mg/L),Antimony (µg/L),...,Thallium (µg/L),Total Kjeldahl Nitrogen (mg/L),Total Organic Carbon (mg/L),Total Phosphorus (mg/L),Total Solids (mg/L),Total Volatile Solids (mg/L),Turbidity (NTU),Vanadium (µg/L),Water Temperature (°C),Zinc (µg/L)
0,Junk Bay,Tseng Lan Shue Stream,JR11,29/04/1986,1,9.7,0,3.3,0.11,NaN,...,NaN,4.1,<1,2.5,210,160,4.1,NaN,25.0,40
1,Junk Bay,Tseng Lan Shue Stream,JR11,19/05/1986,1,5.6,0,3.1,0.11,NaN,...,NaN,5.1,5,2,180,140,4.2,NaN,26.6,60
2,Junk Bay,Tseng Lan Shue Stream,JR11,18/06/1986,1,9,0,1.5,0.13,NaN,...,NaN,3.1,1,1.9,150,110,5.5,NaN,28.4,30
3,Junk Bay,Tseng Lan Shue Stream,JR11,24/07/1986,1,12.2,40,1.8,0.26,NaN,...,NaN,5.5,<1,2.1,34,2,6.5,NaN,26.5,20
4,Junk Bay,Tseng Lan Shue Stream,JR11,15/08/1986,1,8.8,30,1.5,0.73,NaN,...,NaN,5.3,5,4.5,160,82,6.7,NaN,30.1,<10


The `df` DataFrame has been saved to `/content/drive/MyDrive/Water Quality Dataset/Test acc/Dataset 4/processed_water_quality_data.csv`.

In [7]:
columns = ['Conductivity (µS/cm)', 'pH', 'Turbidity (NTU)', 'Water Temperature (°C)']
df = File1[columns]
df.head()

,Conductivity (µS/cm),pH,Turbidity (NTU),Water Temperature (°C)
0,320.0,6.9,4.1,25.0
1,220.0,7.1,4.2,26.6
2,220.0,7.2,5.5,28.4
3,185.0,7.3,6.5,26.5
4,250.0,7.3,6.7,30.1


In [6]:
df['date'] = pd.to_datetime(df['Dates'])
df['year'] = df['date'].dt.year
df = df[df['year'] >= 2010]

KeyError: 'Dates'

In [8]:
df.isnull().sum()

Conductivity (µS/cm)       5
pH                        12
Turbidity (NTU)           16
Water Temperature (°C)     9
dtype: int64

In [9]:
df.describe()

,Conductivity (µS/cm),pH,Turbidity (NTU),Water Temperature (°C)
count,31860.000000,31853.000000,31849.000000,31856.000000
mean,4791.369052,7.484403,27.736262,24.125330
std,9815.923211,0.574610,120.875125,4.603813
min,2.000000,2.800000,-1.100000,6.300000
25%,127.000000,7.200000,2.900000,20.600000
50%,294.000000,7.400000,6.200000,24.700000
75%,2130.250000,7.700000,18.000000,27.800000
max,98800.000000,12.100000,7920.000000,37.100000


In [14]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.isna().sum()

Conductivity (µS/cm)       5
pH                        12
Turbidity (NTU)           16
Water Temperature (°C)     9
dtype: int64

In [15]:
df_cleaned = df.dropna()
df = df_cleaned
df.isna().sum()

Conductivity (µS/cm)      0
pH                        0
Turbidity (NTU)           0
Water Temperature (°C)    0
dtype: int64

In [16]:

print(f"Number of duplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()

Number of duplicate rows: 0


In [17]:
def get_nlwqs_class(ph, turb, cond, temp, normal_temp=31.0):

    # Category A
    if (6.5 <= ph <= 8.5) and (cond <= 1000) and (temp <= normal_temp) and (turb <= 40):
        return 1
    
    # Category B
    elif (6.5 <= ph <= 8.5) and (cond <= 1000) and (temp <= normal_temp) and (40 <= turb <= 170):
        return 2
    
    # Category C
    elif (6.0 <= ph <= 9.0) and (cond <= 2000) and (temp <= normal_temp) and (turb <= 70):
        return 3
    
    # Category D
    elif (5.5 <= ph <= 9.0) and (cond <= 5000) and (temp <= normal_temp) and (turb <= 250):
        return 4
    
    else:
        return 5
    

In [18]:
df['overall_class_nlwqs'] = df.apply(lambda row: get_nlwqs_class(row['pH'], row['Turbidity (NTU)'], row['Conductivity (µS/cm)'], row['Water Temperature (°C)']), axis=1)

In [19]:
df[df['overall_class_nlwqs'] == 5].tail()


,Conductivity (µS/cm),pH,Turbidity (NTU),Water Temperature (°C),overall_class_nlwqs
31801,309.0,8.8,8.0,34.1,5
31840,502.0,8.0,2.7,31.5,5
31841,567.0,7.6,5.2,31.7,5
31842,604.0,7.7,64.0,31.6,5
31860,535.0,7.5,134.2,31.1,5


In [20]:
overall_class_counts = df['overall_class_nlwqs'].value_counts()
display(overall_class_counts)

overall_class_nlwqs
1    17411
5     8684
2     2303
3     1770
4     1660
Name: count, dtype: int64

In [21]:
df[df['overall_class_nlwqs'] == 1]

,Conductivity (µS/cm),pH,Turbidity (NTU),Water Temperature (°C),overall_class_nlwqs
0,320.0,6.9,4.1,25.0,1
1,220.0,7.1,4.2,26.6,1
2,220.0,7.2,5.5,28.4,1
3,185.0,7.3,6.5,26.5,1
4,250.0,7.3,6.7,30.1,1
...,...,...,...,...,...
31857,851.0,7.1,36.6,22.5,1
31858,767.0,7.1,27.0,28.1,1
31859,675.0,7.6,15.3,29.8,1
31861,995.0,7.7,18.5,30.3,1


In [22]:
df.head(10)

,Conductivity (µS/cm),pH,Turbidity (NTU),Water Temperature (°C),overall_class_nlwqs
0,320.0,6.9,4.1,25.0,1
1,220.0,7.1,4.2,26.6,1
2,220.0,7.2,5.5,28.4,1
3,185.0,7.3,6.5,26.5,1
4,250.0,7.3,6.7,30.1,1
5,145.0,6.6,3.2,27.3,1
6,175.0,6.8,2.0,26.0,1
7,350.0,7.2,2.3,19.7,1
8,360.0,7.7,5.0,18.6,1
9,400.0,6.2,24.0,23.3,3


In [23]:
features = ['Conductivity (µS/cm)', 'pH', 'Turbidity (NTU)', 'Water Temperature (°C)']
X = df[features].values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['overall_class_nlwqs'])
unique, counts = np.unique(y, return_counts=True)



In [25]:
warnings.filterwarnings('ignore')

n_folds = 5
n_iter = 20
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

random_forest = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')
random_forest_param = { 'classifier__n_estimators': randint(50, 200), 'classifier__max_depth': [5, 10, 15, None], 'classifier__min_samples_split': randint(2, 15), 'classifier__min_samples_leaf': randint(1, 5)}

xgboost = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
xgboost_param = { 'classifier__n_estimators': randint(50, 200), 'classifier__max_depth': randint(3, 8), 'classifier__learning_rate': uniform(0.01, 0.2)}

svm = SVC(probability=True, random_state=42,class_weight='balanced')
svm_param = { 'classifier__C': uniform(0.1, 10), 'classifier__gamma': ['scale', 'auto'], 'classifier__kernel': ['rbf']}

knn = KNeighborsClassifier(n_jobs=-1)
knn_param = { 'classifier__n_neighbors': randint(3, 15), 'classifier__weights': ['uniform', 'distance'] }

mlp_shallow = MLPClassifier(random_state=42, max_iter=300, early_stopping=True)
mlp_shallow_param = { 'classifier__hidden_layer_sizes': [(50,), (100,), (50, 25)], 'classifier__activation': ['relu', 'tanh'], 'classifier__alpha': uniform(0.0001, 0.01) }

mlp_deep = MLPClassifier(random_state=42, max_iter=300, early_stopping=True)
mlp_deep_param = { 'classifier__hidden_layer_sizes': [(128, 64, 32), (256, 128, 64)], 'classifier__activation': ['relu'], 'classifier__alpha': uniform(0.001, 0.05) }

models = ['Random Forest', 'XGBoost', 'SVM', 'KNN', 'MLP (Shallow)', 'DNN (Deep)']
model_objects = [random_forest, xgboost, svm, knn, mlp_shallow, mlp_deep]
model_parameters = [random_forest_param, xgboost_param, svm_param, knn_param, mlp_shallow_param, mlp_deep_param]

In [26]:
best_models = {}

for i in range(len(models)):
   print("\nTraining " + models[i])
   pipeline = ImbPipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('classifier', model_objects[i])])
   random_search = RandomizedSearchCV(
        estimator=pipeline, 
        param_distributions=model_parameters[i], 
        n_iter=n_iter, 
        cv=skf, 
        scoring='f1_weighted', 
        random_state=42, 
        n_jobs=-1, 
        verbose=0
    )
   
   random_search.fit(X, y)
   best_models[models[i]] = random_search.best_estimator_
   y_pred_cv = cross_val_predict(best_models[models[i]], X, y, cv=skf)
   y_proba_cv = cross_val_predict(best_models[models[i]], X, y, cv=skf, method='predict_proba')
   acc = accuracy_score(y, y_pred_cv)
   prec = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
   rec = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
   f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
   auc_score = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
   print(f"Best params: {random_search.best_params_}")
   print(f"Accuracy: {acc}")
   print(f"Precision: {prec}")
   print(f"Recall: {rec}")
   print(f"F1-Score: {f1}")
   print(f"AUC-ROC: {auc_score}")

save_dir = "dataset_2_classification"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

ensemble_package = {}
ensemble_package['models'] = best_models
ensemble_package['features'] = features
ensemble_package['encoder'] = label_encoder
unified_path = f"{save_dir}/dataset_2_classification.pkl"
joblib.dump(ensemble_package, unified_path)
print("Saved to " + unified_path)


Training Random Forest
Best params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 133}
Accuracy: 0.9995601357295463
Precision: 0.9995616591836354
Recall: 0.9995601357295463
F1-Score: 0.999560372159503
AUC-ROC: 0.9999997922739488

Training XGBoost
Best params: {'classifier__learning_rate': np.float64(0.05820509320520235), 'classifier__max_depth': 6, 'classifier__n_estimators': 57}
Accuracy: 0.9885635289682041
Precision: 0.9888044102991016
Recall: 0.9885635289682041
F1-Score: 0.9886390855009711
AUC-ROC: 0.9999036694823403

Training SVM
Best params: {'classifier__C': np.float64(9.83755518841459), 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Accuracy: 0.9780382053537765
Precision: 0.9795064678652631
Recall: 0.9780382053537765
F1-Score: 0.978467372228759
AUC-ROC: 0.9996471490468725

Training KNN
Best params: {'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
Accuracy: 0.954851074

In [27]:
warnings.filterwarnings('ignore')

anomaly_labels = []
for value in df['overall_class_nlwqs']:
    if value == 5:
        anomaly_labels.append(1)
    else:
        anomaly_labels.append(0)

df['is_anomaly'] = anomaly_labels
y = df['is_anomaly'].values

In [31]:
print(y)

[0 0 0 ... 0 0 0]


In [29]:
n_folds = 10
n_iter = 30
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest_param = { 'classifier__n_estimators': randint(50, 300), 'classifier__max_depth': [5, 10, 15, None], 'classifier__min_samples_split': randint(2, 15), 'classifier__min_samples_leaf': randint(1, 10), 'classifier__max_features': ['sqrt', 'log2', None]}

xgboost = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
xgboost_param = { 'classifier__n_estimators': randint(50, 300), 'classifier__max_depth': randint(3, 10), 'classifier__learning_rate': uniform(0.01, 0.3), 'classifier__subsample': uniform(0.6, 0.4), 'classifier__colsample_bytree': uniform(0.6, 0.4)}

svm = SVC(probability=True, random_state=42)
svm_param = { 'classifier__C': uniform(0.1, 10), 'classifier__gamma': ['scale', 'auto'], 'classifier__kernel': ['rbf']}

knn = KNeighborsClassifier(n_jobs=-1)
knn_param = { 'classifier__n_neighbors': randint(3, 15), 'classifier__weights': ['uniform', 'distance'], 'classifier__p': [1, 2]}

mlp_shallow = MLPClassifier(random_state=42, max_iter=300, early_stopping=True)
mlp_shallow_param = { 'classifier__hidden_layer_sizes': [(50,), (100,)], 'classifier__activation': ['relu', 'tanh'], 'classifier__alpha': uniform(0.0001, 0.01) }

mlp_deep = MLPClassifier(random_state=42, max_iter=500, early_stopping=True)
mlp_deep_param = { 'classifier__hidden_layer_sizes': [(128, 64, 32), (256, 128, 64)], 'classifier__activation': ['relu'], 'classifier__alpha': uniform(0.001, 0.01) }

models1 = ['Random Forest', 'XGBoost', 'SVM', 'KNN', 'MLP (Shallow)', 'DNN (Deep)']
model_objects1 = [random_forest, xgboost, svm, knn, mlp_shallow, mlp_deep]
model_parameters1 = [random_forest_param, xgboost_param, svm_param, knn_param, mlp_shallow_param, mlp_deep_param]

In [32]:
best_models1 = {}
all_f1_scores = {}
all_metrics = {}
print("\nStarting anomaly detection training...")
for i in range(len(models1)):
   print("\nTraining " + models1[i])
   pipeline = ImbPipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('classifier', model_objects1[i])])
   random_search = RandomizedSearchCV(
        estimator=pipeline, 
        param_distributions=model_parameters1[i], 
        n_iter=n_iter, 
        cv=skf, 
        scoring='f1', 
        random_state=42, 
        n_jobs=-1, 
        verbose=0
    )
   
   random_search.fit(X, y)
   best_models1[models1[i]] = random_search.best_estimator_
   fold_scores = cross_val_score(random_search.best_estimator_, X, y, cv=skf, scoring='f1')
   all_f1_scores[models1[i]] = fold_scores
   
   y_pred_cv = cross_val_predict(best_models1[models1[i]], X, y, cv=skf, method='predict')
   y_proba_cv = cross_val_predict(best_models1[models1[i]], X, y, cv=skf, method='predict_proba')

   y_proba_positive = y_proba_cv[:, 1]
   acc = accuracy_score(y, y_pred_cv)
   prec = precision_score(y, y_pred_cv, zero_division=0)
   rec = recall_score(y, y_pred_cv, zero_division=0)
   f1 = f1_score(y, y_pred_cv, zero_division=0)
   f1_std = fold_scores.std()
   roc_auc = roc_auc_score(y,  y_proba_positive)
   pr_auc = average_precision_score(y, y_proba_positive)
   all_metrics[models1[i]] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'F1-Std': f1_std,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    }
   
   print(f"{models1[i]} Finished")
   print(all_metrics[models1[i]])

save_dir = "dataset_2_anomaly"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

ensemble_package = {}
ensemble_package['models'] = best_models1
ensemble_package['features'] = features
ensemble_package['encoder'] = label_encoder
unified_path = save_dir + "/dataset_2_anomaly.pkl"
joblib.dump(ensemble_package, unified_path)
print("Saved to " + unified_path)


Starting anomaly detection training...

Training Random Forest
Random Forest Finished
{'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'F1-Std': np.float64(0.0), 'ROC-AUC': 1.0, 'PR-AUC': 1.0}

Training XGBoost
XGBoost Finished
{'Accuracy': 0.9977692597712706, 'Precision': 0.9958549222797928, 'Recall': 0.9959695992630124, 'F1-Score': 0.9959122574702055, 'F1-Std': np.float64(0.002652343509853557), 'ROC-AUC': 0.9999729653073868, 'PR-AUC': 0.9999266573007008}

Training SVM
SVM Finished
{'Accuracy': 0.9951300741485485, 'Precision': 0.9839972761321076, 'Recall': 0.998387839705205, 'F1-Score': 0.9911403258073735, 'F1-Std': np.float64(0.0021552133310150426), 'ROC-AUC': 0.9999133904675959, 'PR-AUC': 0.9997688906683362}

Training KNN
KNN Finished
{'Accuracy': 0.992867915043358, 'Precision': 0.9817705366298279, 'Recall': 0.9922846614463381, 'F1-Score': 0.9869995991065804, 'F1-Std': np.float64(0.00292671277121441), 'ROC-AUC': 0.9994718296263968, 'PR-AUC': 0.9988548461317631}

